# Análise Exploratória

Visualizações interativas (Plotly) dos dados Gold.

**Pré-requisito:** `03_gold.ipynb` já executado (gold populado).

| Seção | Conteúdo |
|---|---|
| 5.1 | Evolução do gap salarial por UF e trimestre (PNAD) |
| 5.2 | Saldo de empregos por sexo e UF (CAGED) |
| 5.3 | Gap salarial por nível de escolaridade (RAIS) |
| 5.4 | Participação feminina por setor (RAIS) |
| 5.5 | Gap salarial por setor (RAIS) |

In [1]:
import sys, pathlib

_root = pathlib.Path.cwd()
if not (_root / "config.py").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [2]:
from config import config
import pandas as pd
import plotly.express as px

## 5.1 — Evolução do Gap Salarial por UF (PNAD)

In [3]:
df_gap = pd.read_csv(config.gold_dir / "gap_salarial_por_uf_sexo_idade.csv", encoding="utf-8")

_UF_NOMES = {41: "Paraná", 42: "Santa Catarina", 43: "Rio Grande do Sul"}
df_gap["uf"] = df_gap["uf_cod"].apply(lambda x: _UF_NOMES.get(int(x), str(x)))

def _fmt_periodo(p: int) -> str:
    s = str(int(p))
    return f"T{int(s[4:])} {s[:4]}"

df_gap["periodo_label"] = df_gap["periodo"].apply(_fmt_periodo)

df_gap_uf = (
    df_gap.groupby(["uf", "periodo", "periodo_label"])["gap_pct"]
    .mean()
    .reset_index()
    .sort_values("periodo")
)

fig = px.line(
    df_gap_uf,
    x="periodo_label",
    y="gap_pct",
    color="uf",
    title="Evolução do Gap Salarial de Gênero — Região Sul (%)",
    labels={"gap_pct": "Gap (%)", "periodo_label": "Trimestre", "uf": "Estado"},
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Paridade salarial")
fig.update_layout(
    xaxis_tickangle=45,
    xaxis_title="Trimestre",
    yaxis_title="Gap salarial (%)",
    legend_title_text="Estado",
    hovermode="x unified",
)
fig.show()

## 5.2 — Saldo de Empregos por Sexo e UF (CAGED)

In [4]:
df_vagas = pd.read_csv(config.gold_dir / "vagas_saldo_por_uf_perfil.csv", encoding="utf-8")
df_vagas = df_vagas[df_vagas["sexo"].isin(["Masculino", "Feminino"])]

df_saldo_sexo = (
    df_vagas.groupby(["ano", "sigla_uf", "sexo"])["saldo"]
    .sum()
    .reset_index()
)
df_saldo_sexo["ano"] = df_saldo_sexo["ano"].astype(str)

fig2 = px.bar(
    df_saldo_sexo,
    x="ano",
    y="saldo",
    color="sexo",
    color_discrete_map={"Masculino": "#4C72B0", "Feminino": "#DD8452"},
    barmode="group",
    facet_col="sigla_uf",
    facet_col_spacing=0.08,
    title="Saldo Líquido de Empregos por Sexo e UF (CAGED)",
    labels={"saldo": "Saldo líquido", "ano": "Ano", "sexo": "Sexo"},
)
fig2.update_layout(legend_title_text="Sexo", yaxis_title="Saldo líquido de empregos")
fig2.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig2.show()

## 5.3 — Gap Salarial por Nível de Escolaridade (RAIS)

In [5]:
from analise.gap_salarial import ESCOLARIDADE_ORDEM

df_esc = pd.read_csv(config.gold_dir / "gap_salarial_por_escolaridade.csv", encoding="utf-8")

df_esc_agg = (
    df_esc.groupby("escolaridade")[["salario_masculino", "salario_feminino", "gap_pct"]]
    .mean()
    .round(2)
    .reset_index()
)
df_esc_agg["escolaridade"] = pd.Categorical(
    df_esc_agg["escolaridade"], categories=ESCOLARIDADE_ORDEM, ordered=True
)
df_esc_agg = df_esc_agg.sort_values("escolaridade")

print("Gap salarial por nível de escolaridade — Região Sul (RAIS)")
display(df_esc_agg.rename(columns={
    "escolaridade": "Escolaridade",
    "salario_masculino": "Salário médio M (R$)",
    "salario_feminino": "Salário médio F (R$)",
    "gap_pct": "Gap (%)",
}))

fig3 = px.bar(
    df_esc_agg,
    x="escolaridade",
    y="gap_pct",
    title="Gap Salarial de Gênero por Nível de Escolaridade — Região Sul (RAIS)",
    labels={"gap_pct": "Gap (%)", "escolaridade": "Escolaridade"},
    color="gap_pct",
    color_continuous_scale="RdYlGn_r",
    text="gap_pct",
)
fig3.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig3.update_layout(
    xaxis_title="Nível de escolaridade",
    yaxis_title="Gap salarial (%)",
    coloraxis_showscale=False,
    xaxis={"categoryorder": "array", "categoryarray": ESCOLARIDADE_ORDEM},
)
fig3.show()

Gap salarial por nível de escolaridade — Região Sul (RAIS)


,Escolaridade,Salário médio M (R$),Salário médio F (R$),Gap (%)
0,Analfabeto,2295.65,1745.43,31.45
2,Fund. incompleto,2385.41,1626.31,46.25
1,Fund. completo,2541.86,1753.16,45.01
4,Médio incompleto,2260.84,1656.96,36.36
3,Médio completo,2746.85,2098.58,30.83
7,Superior incompleto,3691.12,2550.58,44.25
6,Superior completo,7913.54,5043.73,56.29
5,Pós-graduação,11651.63,8721.37,33.54


## 5.4 — Participação Feminina por Setor (RAIS)

In [6]:
df_setor = pd.read_csv(config.gold_dir / "gap_salarial_por_setor.csv", encoding="utf-8")

df_fem = df_setor[df_setor["sexo"] == "Feminino"].copy()
df_setor_agg = (
    df_fem.groupby("setor")[["share_pct", "gap_pct", "salario_medio"]]
    .mean()
    .round(2)
    .sort_values("share_pct")
    .reset_index()
)

fig4 = px.bar(
    df_setor_agg,
    x="share_pct",
    y="setor",
    orientation="h",
    title="Participação feminina por setor de atividade (%) — Região Sul (RAIS)",
    labels={"share_pct": "Participação feminina (%)", "setor": "Setor"},
    color="share_pct",
    color_continuous_scale="RdYlGn",
    text="share_pct",
)
fig4.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig4.add_vline(x=50, line_dash="dash", line_color="gray", annotation_text="50% — paridade")
fig4.update_layout(
    xaxis_title="Participação feminina (%)",
    coloraxis_showscale=False,
    xaxis_range=[0, 100],
)
fig4.show()

## 5.5 — Gap Salarial por Setor (RAIS)

In [7]:
fig5 = px.bar(
    df_setor_agg.dropna(subset=["gap_pct"]).sort_values("gap_pct", ascending=False),
    x="gap_pct",
    y="setor",
    orientation="h",
    title="Gap Salarial de Gênero por Setor (%) — Região Sul (RAIS)",
    labels={"gap_pct": "Gap salarial (%)", "setor": "Setor"},
    color="gap_pct",
    color_continuous_scale="RdYlGn_r",
    text="gap_pct",
)
fig5.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig5.add_vline(x=0, line_dash="dash", line_color="gray")
fig5.update_layout(xaxis_title="Gap salarial (%)", coloraxis_showscale=False)
fig5.show()

print("\nResumo: setores com maior gap salarial e menor participação feminina")
display(
    df_setor_agg[["setor", "share_pct", "gap_pct", "salario_medio"]]
    .rename(columns={
        "setor": "Setor",
        "share_pct": "Part. feminina (%)",
        "gap_pct": "Gap salarial (%)",
        "salario_medio": "Salário médio F (R$)",
    })
    .sort_values("Gap salarial (%)", ascending=False)
)


Resumo: setores com maior gap salarial e menor participação feminina


,Setor,Part. feminina (%),Gap salarial (%),Salário médio F (R$)
13,Outros serviços,58.07,68.35,2381.06
14,Financeiro,58.18,64.79,6009.40
8,Artes e cultura,46.45,51.76,1888.27
18,Adm. pública,65.25,44.02,4821.75
6,Ind. transformação,35.60,43.41,2362.00
19,Educação,66.44,40.41,4048.98
10,Administrativo,50.11,39.41,1848.34
20,Saúde,81.20,38.96,3127.92
7,TIC,36.68,37.10,3830.09
3,Eletricidade e gás,19.74,32.70,6777.07
